In [ ]:
# import statements

import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from lmfit import Model, models, minimize, Minimizer, create_params
from skopt import gp_minimize

%matplotlib widget


In [ ]:
# global variables

def calibrate_kernel_size(type, val_min, val_max):
    """
    yea gotta change this

    Parameter wd_val_min: the minimum working distance (float) that the SEM can image at
    """

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    sums = -1
    kernel_size = 5

    if type == 'wd':
        phenom.SetSemWD(val_min)

    elif type == "xst":
        phenom.SetSemStigmate(ppi.Position(val_min, 0))

    elif type == "yst":
        phenom.SetSemStigmate(ppi.Position(0, val_min))

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    while np.round(sums0, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) 
        filtered = cv2.Canny(blurred_img, 50, 150)
        sums = filtered.sum()

        if np.round(sums, 4) != 0:
            kernel_size = kernel_size + 2

    return kernel_size


wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -0.2 #-1
xst_val_max = 0.2 #1
yst_val_min = -0.2 #-1
yst_val_max = 0.2 #1

size = calibrate_kernel_size('wd', wd_val_min, wd_val_max)
x_size = calibrate_kernel_size('xst', xst_val_min, xst_val_max)
y_size = calibrate_kernel_size('yst', yst_val_min, yst_val_max)



In [ ]:
# autocorrelation functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def filtered_autocorr(img_array, kernel_size):
    """
    Returns the autocorrelation array for the given image array and the kernel size to blur the image.

    Parameter img_array: a Numpy array
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """
    
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = blurred_img
    # filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    return acdata

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, params = pars) # uses the above parameters and  model to fit the data

    # plt.figure()
    # plt.imshow(ac)
    # plt.contour(result.best_fit)

    return result

In [ ]:
# optimization functions

def optimum_wd(wd, kernel_size = size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the center pixel given pars, 
    a Parameters dictionary containing the working distance range to calculate over.
    The objective function to minimize for working distance.

    Parameter phenom: a Phenom object
    Parameter wd: a float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    print(wd)
    phenom.SetSemWD(wd[0])
    # print(kernel_size)

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    
    acdata = filtered_autocorr(img_array, kernel_size)

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)


def auto_focus(wd_val_min, wd_val_max):
    """
    Sets the working distance to the value corresponding to a focused image.

    Parameter wd_val_min: the minimum working distance (float) that is considered when finding the optimal working distance
    Parameter wd_val_max: the maximum working distance (float) that is considered when finding the optimal working distance
    """

    res = gp_minimize(optimum_wd, 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", n_calls = 35, 
                      n_initial_points = 5, 
                      verbose = True, 
                      random_state = 1234) 

    print(res.x[0])
    phenom.SetSemWD(res.x[0])
    return res.x[0]


def optimum_xstig(xst, kernel_size = x_size):
    print(xst)
    phenom.SetSemStigmate(ppi.Position(xst[0], 0))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        print(threshold, acdata.max(), acdata.min())
        acdata[acdata < threshold] = 0

        if acdata.sum() != 0:
            acdata /= acdata.sum()

    result = incoh_psf_fit(acdata, -2, .3)

    return result.best_values['stig']

    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    # acdata = acdata - np.min(acdata) # subtracts the background noise
    # acdata /= acdata.sum() # normalizes to help with background noise


def optimum_ystig(yst, kernel_size = y_size):
    phenom.SetSemStigmate(ppi.Position(0, yst[0]))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        print(threshold, acdata.max(), acdata.min())
        acdata[acdata < threshold] = 0

        if acdata.sum() != 0:
            acdata /= acdata.sum()

    result = incoh_psf_fit(acdata, -2, .3)

    return result.best_values['stig']

    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    # acdata = acdata - np.min(acdata) # subtracts the background noise
    # acdata /= acdata.sum() # normalizes to help with background noise


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    x_res = gp_minimize(optimum_xstig, 
                        [(xst_val_min, xst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 35, 
                        n_initial_points = 5, 
                        verbose = True, 
                        random_state = 1234) 

    y_res = gp_minimize(optimum_ystig, 
                        [(yst_val_min, yst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 35, 
                        n_initial_points = 5, 
                        verbose = True, 
                        random_state = 1234) 
                        
    print(x_res.x[0], y_res.x[0])

    phenom.SetSemStigmate(ppi.Position(x_res.x[0], y_res.x[0]))

    # return x_res.x[0], y_res.x[0]

In [ ]:
# connect to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

In [ ]:
# main thing 

auto_focus(wd_val_min, wd_val_max)

def whole_thingy():
    prev = None # wd value probably?
    curr = None

    magnification = 10000

    while prev is None or np.abs(curr - prev) > 0.001
        # set sem magnification to magnification

        if magnification < 40000:
            prev = autofocus(wd_val_min, wd_val_max)
        else:
            prev = autofocus(wd_val_min, wd_val_max)
            auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

        magnification *= 2
        curr = prev

# loc 1, 10k: 9.742, 9.634, 9.770
# loc 1, 20k: 9.834, 9.683, 9.642
# loc 1, 30k: 9.714, 9.679, 9.695
# loc 1, 40k: 9.679, 9.705, 9.512

# auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

# loc 1, 40k: -0.10953524713198484 -0.2 very wrong lmao

In [ ]:
wds = [0.002, 0.00968, 0.010]

xdim = 256 # x and y dimensions of image taken
ydim = 256
stack = 1
HDR = False
scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

for wd in wds:
    phenom.SetSemWD(wd)

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (11, 11), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    acdata = acdata[128 -  16: 128 + 16, 128 - 16: 128 + 16]

    if acdata.sum() != 0:
        acdata = acdata - np.min(acdata)
        acdata /= acdata.sum()

    result = incoh_psf_fit(acdata, -2, .3)

    print(result.best_values)

    plt.figure()

    plt.subplot(1, 3, 1)
    plt.imshow(filtered_data)
    plt.colorbar()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)
    plt.colorbar()

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)
    plt.colorbar()

    plt.show()



In [ ]:
xdim = 256 # x and y dimensions of image taken
ydim = 256
stack = 8
HDR = False
scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))

#phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata.mean()
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))

In [ ]:
# covariance function, probably dont need

def covariance_matrix(ac):
    """
    Returns the computed covariance matrix given the auto_correlation data.

    Parameter ac: a 2d array with values corresponding to the auto_correlation data at each position i,j
    """

    combine = np.array(np.meshgrid(np.linspace(0, ac.shape[0], ac.shape[0]), # creates 2d array with vector i,j
                                   np.linspace(0, ac.shape[1], ac.shape[1]))).T.reshape(-1, 256, 2)
    
    # covariance equation found from:
    # https://math.stackexchange.com/questions/1939040/get-covariance-of-pixels-in-an-image

    n = 0
    m = 0
    coeff1 = 0
    coeff2 = 0

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            n += val
            m += val ** 2
            coeff1 += val * np.atleast_2d(combine[row][col])

    mean = (1 / n) * coeff1

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            diff = np.atleast_2d(combine[row][col]) - mean
            tpose = diff.T
            coeff2 += val * (diff * tpose)

    cov = (n / (n ** 2 - m)) * coeff2

    return {"covarxx": cov[0][0], "covaryy": cov[1][1], "covarxy": cov[0][1]}

In [ ]:
# optimization with lmfit functions

def optimum_wd_lmfit(pars):
    parvals = pars.valuesdict()
    phenom.SetSemWD(parvals['wd'])
    print(parvals['wd'])

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered
    kernel_size = parvals['size']

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    sums = calculate_sums(img_array, kernel_size)
    
    # acdata = acdata - np.min(acdata)
    # acdata /= acdata.sum()

    # fitted_vals = incoh_psf_fit(acdata, -2, .3).best_values

    # fit_def.append(fitted_vals['defocus'])
    # return fitted_vals["defocus"]
    
    return sums * (-1)


def auto_focus_lmfit(wd_val_min, wd_val_max, res):
    kernel_size = calibrate_kernel_size('wd', wd_val_min, wd_val_max)
    print(kernel_size)
    
    # while res >= 0.001:
    params = create_params(wd = dict(value = wd_val_min, vary = True, min = wd_val_min, max = wd_val_max, brute_step = res / 1000),
                        size = dict(value = kernel_size, vary = False))

    fitter = Minimizer(optimum_wd_lmfit, params)
    result = fitter.minimize(method = 'brute')

    print(result.brute_x0)

    plt.figure()
    plt.plot(result.brute_grid, result.brute_Jout)

        # res /= 10
        # wd_val_min = result.brute_x0 - (res / 100)
        # wd_val_max = result.brute_x0 + (res / 100)

    phenom.SetSemWD(result.brute_x0)


def optimum_xstig_lmfit(pars):
    parvals = pars.valuesdict()
    phenom.SetSemStigmate(ppi.Position(parvals["xst"], 0))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, 256, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    fit_stig.append(fitted_vals['stig'])

    return fitted_vals["stig"]


def optimum_ystig_lmfit(pars):
    parvals = pars.valuesdict()
    phenom.SetSemStigmate(ppi.Position(parvals["xst"], parvals["yst"]))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, 256, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    fit_stig.append(fitted_vals['stig'])

    return fitted_vals["stig"]


def auto_stig_lmfit(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    xparams = create_params(xst = dict(value = xst_val_min, vary = True, min = xst_val_min, max = xst_val_max, brute_step = 0.01))

    xfitter = Minimizer(optimum_xstig, xparams)
    xresult = xfitter.minimize(method = 'brute')

    yparams = create_params(xst = dict(value = xresult.brutex0, vary = False), yst = dict(value = yst_val_min, vary = True, min = yst_val_min, max = yst_val_max, brute_step = 0.01))

    yfitter = Minimizer(optimum_ystig, yparams)
    yresult = yfitter.minimize(method = 'brute')

    plt.figure()
    plt.plot(result.brute_grid, result.brute_Jout)

In [ ]:
wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -1
xst_val_max = 1
yst_val_min = -1
yst_val_max = 1
res = .1

fit_stig = []

auto_focus_lmfit(wd_val_min, wd_val_max, res)

In [ ]:
# OLD optimization functions (blur + fit for wd)

def optimum_wd(wd, kernel_size = size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the center pixel given pars, 
    a Parameters dictionary containing the working distance range to calculate over.
    The objective function to minimize for working distance.

    Parameter phenom: a Phenom object
    Parameter wd: a float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    print(wd)
    phenom.SetSemWD(wd[0])
    # print(kernel_size)

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    
    acdata = filtered_autocorr(img_array, kernel_size)

    # acdata = acdata[128 -  16: 128 + 16, 128 - 16: 128 + 16]

    # if acdata.sum() != 0:
    #     acdata = acdata - np.min(acdata)
    #     acdata /= acdata.sum()

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        print(threshold, acdata.max(), acdata.min())
        acdata[acdata < threshold] = 0

        if acdata.sum() != 0:
            acdata /= acdata.sum()

    result = incoh_psf_fit(acdata, -2, .3)

    return result.best_values['defocus']

    # sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    # return sums * (-1)


def auto_focus(wd_val_min, wd_val_max):
    """
    Sets the working distance to the value corresponding to a focused image.

    Parameter wd_val_min: the minimum working distance (float) that is considered when finding the optimal working distance
    Parameter wd_val_max: the maximum working distance (float) that is considered when finding the optimal working distance
    """

    res = gp_minimize(optimum_wd, 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", n_calls = 35, 
                      n_initial_points = 5, 
                      verbose = True, 
                      random_state = 1234) 

    print(res.x[0])
    phenom.SetSemWD(res.x[0])


def optimum_xstig(xst, kernel_size = x_size):
    print(xst)
    phenom.SetSemStigmate(ppi.Position(xst[0], 0))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    return fitted_vals["stig"]


def optimum_ystig(yst, kernel_size = y_size):
    phenom.SetSemStigmate(ppi.Position(0, yst[0]))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    acdata = filtered_autocorr(img_array, kernel_size) # calculates the filtered auto_correlation data

    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    return fitted_vals["stig"]


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    x_res = gp_minimize(optimum_xstig, 
                        [(xst_val_min, xst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 35, 
                        n_initial_points = 5, 
                        verbose = True, 
                        random_state = 1234) 

    y_res = gp_minimize(optimum_ystig, 
                        [(yst_val_min, yst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 35, 
                        n_initial_points = 5, 
                        verbose = True, 
                        random_state = 1234) 
                        
    print(x_res.x[0], y_res.x[0])

    phenom.SetSemStigmate(ppi.Position(x_res.x[0], y_res.x[0]))